In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem import Descriptors
from tqdm import tqdm
import logging
import time
from func_timeout import func_timeout, FunctionTimedOut

# For modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

# Set up logging for errors during descriptor computation
logging.basicConfig(filename='descriptor_errors.log', level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s')

# =============================================================================
# 1. Load Data and Prepare Molecules
# =============================================================================
data_df = pd.read_csv("./dw_data/Opt1_acidic_tr.csv")
smiles_col = 'OriginalSmiles'
target = 'pKa'

# Create a SaltRemover instance.
saltRemover = SaltRemover(defnFilename='./Salts.txt')

# Convert SMILES to RDKit Mol objects (and strip salts) and store in a new column.
data_df['Mol'] = data_df[smiles_col].astype(str).apply(
    lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s))
)

print(f"Total molecules: {len(data_df)}")
print(f"Valid molecules: {data_df['Mol'].notnull().sum()}")

# Get the list of molecule objects.
mols = data_df['Mol'].tolist()

Total molecules: 2220
Valid molecules: 2220


In [2]:
# =============================================================================
# 2. Define a Function to Compute Filtered Descriptors for a Molecule
# =============================================================================
def safe_call(func, mol, timeout=1):
    """
    Calls a function with a timeout using func_timeout.
    Returns the result if completed within 'timeout' seconds,
    otherwise returns np.nan.
    """
    try:
        result = func_timeout(timeout, func, args=(mol,))
        return result
    except FunctionTimedOut:
        logging.info(f"Timeout in {func.__name__}")
        return np.nan
    except Exception as e:
        logging.info(f"Error in {func.__name__}: {e}")
        return np.nan

def compute_descriptors_for_mol(mol):
    """
    Computes all RDKit descriptors for a given molecule,
    excluding descriptors known to cause errors.
    Each descriptor call is wrapped in safe_call() with a timeout.
    """
    import numpy as np
    from rdkit import Chem
    from rdkit.Chem import Descriptors

    # Define descriptors to exclude
    error_descriptors = {
        "MaxEStateIndex", "MinEStateIndex", 
        "MaxAbsEStateIndex", "MinAbsEStateIndex",
        "FpDensityMorgan1", "FpDensityMorgan2", "FpDensityMorgan3"
    }
    filtered_desc_funcs = {name: func for name, func in Descriptors.descList 
                             if name not in error_descriptors}
    
    if mol is None:
        return None
    elif Chem.MolToSmiles(mol) == '':
        return None
    
    desc_values = {}
    for name, func in filtered_desc_funcs.items():
        value = safe_call(func, mol, timeout=1)
        desc_values[name] = value
    return desc_values

# =============================================================================
# 3. Compute Descriptors Serially (Using func_timeout)
# =============================================================================
desc_data = []
start_time = time.time()
for i, mol in tqdm(enumerate(mols), total=len(mols), desc="Computing filtered descriptors (serial)"):
    try:
        result = compute_descriptors_for_mol(mol)
        if result is not None:
            desc_data.append(result)
    except Exception as e:
        logging.info(f"Molecule {i} error: {e}")
        desc_data.append(None)
end_time = time.time()
print(f"Descriptor computation took {end_time - start_time:.2f} seconds.")

Computing filtered descriptors (serial): 100%|█████████████████████████████████████| 2220/2220 [01:16<00:00, 29.20it/s]

Descriptor computation took 76.04 seconds.


In [3]:
# =============================================================================
# 4. Build DataFrame and Clean It
# =============================================================================
desc_df = pd.DataFrame(desc_data)
# Drop rows that are None or contain any NaN values.
desc_df_clean = desc_df.dropna().reset_index(drop=True)
# Remove descriptor columns with zero variance.
desc_df_clean = desc_df_clean.loc[:, desc_df_clean.std() > 0]

print(f"Final descriptor matrix shape: {desc_df_clean.shape}")
desc_df_clean.to_excel("filtered_descriptors_serial.xlsx", index=False)
print("Filtered descriptors saved to filtered_descriptors_serial.xlsx")

Final descriptor matrix shape: (2196, 190)
Filtered descriptors saved to filtered_descriptors_serial.xlsx


In [4]:

# =============================================================================
# 5. Modeling: Evaluate Each Descriptor Individually Using XGBoost
# =============================================================================
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

def train_and_evaluate_single_descriptor(X, y):
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
    import xgboost as xgb
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=11)
    xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
    grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                               scoring='r2', cv=5, n_jobs=5, verbose=0)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred_test = best_model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    test_mse = mean_squared_error(y_test, y_pred_test)
    test_rmse = np.sqrt(test_mse)
    return {
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_MSE': test_mse,
        'Test_RMSE': test_rmse,
        'Best_Params': str(grid_search.best_params_)
    }

performance_list = []
print("Evaluating each descriptor individually...")
for col in tqdm(desc_df_clean.columns, total=len(desc_df_clean.columns)):
    try:
        X_single = desc_df_clean[[col]].values.astype(float)
        y_vals = data_df.loc[desc_df_clean.index, target].values
        metrics = train_and_evaluate_single_descriptor(X_single, y_vals)
        metrics['Descriptor'] = col
        performance_list.append(metrics)
    except Exception as e:
        logging.info(f"Error evaluating descriptor {col}: {e}")

performance_df = pd.DataFrame(performance_list)
performance_df.sort_values(by='Test_R2', ascending=False, inplace=True)
performance_df.to_excel("individual_descriptor_performance.xlsx", index=False)
print("Individual descriptor performance saved to individual_descriptor_performance.xlsx")
print("\nBest individual descriptors based on Test R2:")
print(performance_df[['Descriptor', 'Test_R2']].head())

# =============================================================================
# 6. Combine the Best Descriptors and Train a Combined Model
# =============================================================================
num_best = 190
best_descriptors = performance_df.head(num_best)['Descriptor'].tolist()
print(f"\nCombining the top {num_best} descriptors: {best_descriptors}")

X_combined = desc_df_clean[best_descriptors].values.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X_combined, data_df.loc[desc_df_clean.index, target].values, test_size=0.25, random_state=11)
xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                           scoring='r2', cv=5, n_jobs=5, verbose=0)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

combined_metrics = {
    'Train_R2': r2_score(y_train, y_pred_train),
    'Test_R2': r2_score(y_test, y_pred_test),
    'Test_MAE': mean_absolute_error(y_test, y_pred_test),
    'Test_MSE': mean_squared_error(y_test, y_pred_test),
    'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    'Best_Params': str(grid_search.best_params_)
}

print("\nCombined model performance using top descriptors:")
for k, v in combined_metrics.items():
    print(f"{k}: {v}")

combined_df = pd.DataFrame([combined_metrics])
combined_df.to_excel("combined_descriptor_model_performance.xlsx", index=False)
print("Combined model performance saved to combined_descriptor_model_performance.xlsx")

Evaluating each descriptor individually...


100%|████████████████████████████████████████████████████████████████████████████████| 190/190 [03:58<00:00,  1.26s/it]


Individual descriptor performance saved to individual_descriptor_performance.xlsx

Best individual descriptors based on Test R2:
             Descriptor   Test_R2
6      MinPartialCharge  0.566609
7   MaxAbsPartialCharge  0.506964
8   MinAbsPartialCharge  0.438733
5      MaxPartialCharge  0.416898
16         BCUT2D_MRLOW  0.396020

Combining the top 190 descriptors: ['MinPartialCharge', 'MaxAbsPartialCharge', 'MinAbsPartialCharge', 'MaxPartialCharge', 'BCUT2D_MRLOW', 'TPSA', 'SMR_VSA1', 'PEOE_VSA14', 'SlogP_VSA2', 'fr_COO', 'fr_COO2', 'EState_VSA1', 'BCUT2D_MWHI', 'SMR_VSA10', 'EState_VSA9', 'PEOE_VSA3', 'EState_VSA8', 'fr_Al_COO', 'VSA_EState5', 'PEOE_VSA10', 'PEOE_VSA1', 'SMR_VSA7', 'VSA_EState3', 'BCUT2D_MRHI', 'PEOE_VSA2', 'PEOE_VSA11', 'SMR_VSA3', 'SlogP_VSA1', 'SlogP_VSA3', 'HallKierAlpha', 'PEOE_VSA9', 'fr_C_O', 'SlogP_VSA6', 'fr_Ar_COO', 'SMR_VSA9', 'EState_VSA10', 'NOCount', 'PEOE_VSA12', 'SlogP_VSA12', 'EState_VSA3', 'PEOE_VSA8', 'fr_phenol', 'MolLogP', 'EState_VSA2', 'fr_phe